## Ejercicio 3: Selección y construcción de variables predictoras

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)

## 3.1 y 3.2. Variables predictoras candidatas

A continuación se define el conjunto **candidato** de variables predictoras,
junto con su tipo y su justificación teórica. Este listado se filtrará más
adelante (sección 3.4) para eliminar cualquier variable que haya sido usada
directa o indirectamente en la construcción del índice de cianobacteria
(Ejercicio 2), evitando así fuga de información (*data leakage*).

In [2]:
predictores_candidatos = [
    # --- Bandas espectrales Sentinel-2 ---
    dict(variable="B02", tipo="Banda espectral",
         representa="Reflectancia en la banda azul (490 nm)",
         justificacion="Sensible a turbidez y materia orgánica disuelta en el agua"),
    dict(variable="B03", tipo="Banda espectral",
         representa="Reflectancia en la banda verde (560 nm)",
         justificacion="Cercana al pico de reflectancia de pigmentos algales (clorofila-a)"),
    dict(variable="B04", tipo="Banda espectral",
         representa="Reflectancia en la banda roja (665 nm)",
         justificacion="Zona de fuerte absorción por clorofila; contraste con B03/B05"),
    dict(variable="B05", tipo="Banda espectral",
         representa="Reflectancia en el borde rojo / red edge (705 nm)",
         justificacion="Muy utilizada en índices de detección de clorofila y cianobacteria (p. ej. NDCI)"),
    dict(variable="B08", tipo="Banda espectral",
         representa="Reflectancia en infrarrojo cercano (842 nm)",
         justificacion="Contraste agua/vegetación; insumo de NDVI y NDWI"),
    dict(variable="B11", tipo="Banda espectral",
         representa="Reflectancia en SWIR 1 (1610 nm)",
         justificacion="Ayuda a distinguir agua de nubes, sombras y tierra"),
    dict(variable="B12", tipo="Banda espectral",
         representa="Reflectancia en SWIR 2 (2190 nm)",
         justificacion="Refuerza la discriminación agua/no-agua junto con B11"),

    # --- Índices espectrales ---
    dict(variable="NDVI", tipo="Índice espectral",
         representa="(B08 - B04) / (B08 + B04)",
         justificacion="Detecta biomasa/vegetación flotante y algas superficiales"),
    dict(variable="NDWI", tipo="Índice espectral",
         representa="(B03 - B08) / (B03 + B08)",
         justificacion="Delimita el cuerpo de agua y su contenido de humedad"),

    # --- Características espaciales ---
    dict(variable="lat", tipo="Espacial",
         representa="Latitud de la observación",
         justificacion="Captura gradientes espaciales (zonas costeras, afluentes, urbanización)"),
    dict(variable="lon", tipo="Espacial",
         representa="Longitud de la observación",
         justificacion="Idem, en la otra dimensión geográfica"),
    dict(variable="lago", tipo="Espacial (categórica)",
         representa="Atitlán o Amatitlán",
         justificacion="Diferencias estructurales/tróficas entre lagos; se excluye en Ejercicio 7 (generalización)"),

    # --- Características temporales ---
    dict(variable="mes", tipo="Temporal",
         representa="Mes de adquisición de la imagen",
         justificacion="Las floraciones de cianobacteria suelen ser estacionales (época cálida/seca)"),

    # --- Variables ambientales externas ---
    dict(variable="temp_prom", tipo="Ambiental externa",
         representa="Temperatura promedio mensual (Weatherspark)",
         justificacion="Temperaturas altas favorecen la proliferación de cianobacterias"),
    dict(variable="precip_prom", tipo="Ambiental externa",
         representa="Precipitación promedio mensual (Weatherspark)",
         justificacion="Relacionada con dilución de nutrientes vs. estancamiento del agua"),
]

df_predictores = pd.DataFrame(predictores_candidatos)
df_predictores


,variable,tipo,representa,justificacion
0,B02,Banda espectral,Reflectancia en la banda azul (490 nm),Sensible a turbidez y materia orgánica disuelta en el agua
1,B03,Banda espectral,Reflectancia en la banda verde (560 nm),Cercana al pico de reflectancia de pigmentos algales (clorofila-a)
2,B04,Banda espectral,Reflectancia en la banda roja (665 nm),Zona de fuerte absorción por clorofila; contraste con B03/B05
3,B05,Banda espectral,Reflectancia en el borde rojo / red edge (705 nm),Muy utilizada en índices de detección de clorofila y cianobacteria (p. ej. NDCI)
4,B08,Banda espectral,Reflectancia en infrarrojo cercano (842 nm),Contraste agua/vegetación; insumo de NDVI y NDWI
5,B11,Banda espectral,Reflectancia en SWIR 1 (1610 nm),"Ayuda a distinguir agua de nubes, sombras y tierra"
6,B12,Banda espectral,Reflectancia en SWIR 2 (2190 nm),Refuerza la discriminación agua/no-agua junto con B11
7,NDVI,Índice espectral,(B08 - B04) / (B08 + B04),Detecta biomasa/vegetación flotante y algas superficiales
8,NDWI,Índice espectral,(B03 - B08) / (B03 + B08),Delimita el cuerpo de agua y su contenido de humedad
9,lat,Espacial,Latitud de la observación,"Captura gradientes espaciales (zonas costeras, afluentes, urbanización)"


**Nota:** este listado es candidato. Una vez definido en el Ejercicio 2 qué
bandas/índice se usaron para construir la variable respuesta (presencia de
cianobacteria), se debe volver a esta tabla y remover las variables que
correspondan (ver sección 3.4).

## 3.3. Ingeniería de características (variables adicionales propuestas)

Se proponen las siguientes variables nuevas, cada una con su justificación.
Las funciones quedan listas para aplicarse sobre el `GeoDataFrame`/`DataFrame`
final una vez esté disponible (Ejercicio 1).

1. **Ratios adicionales de bandas** (que no formen parte del índice de
   cianobacteria) — p. ej. `B05/B04`, capturan información espectral extra
   sin duplicar la variable respuesta.
2. **NDVI/NDWI promediados en una vecindad espacial** (p. ej. radio de 30–50 m)
   — suaviza el ruido propio de un solo píxel.
3. **Codificación cíclica de la fecha** (seno/coseno del día del año) — permite
   capturar estacionalidad sin usar una variable categórica abrupta (mes).
4. **Distancia a la orilla del lago** — zonas someras y cercanas a la costa
   suelen tener mayor acumulación de nutrientes y menor circulación de agua.
5. **Interacción NDVI × NDWI** — resalta píxeles que son simultáneamente
   "agua" y tienen señal de biomasa, un patrón típico de floraciones.

In [3]:
def calcular_ratios_bandas(df, banda_num, banda_den, nombre_nueva):
    df = df.copy()
    df[nombre_nueva] = df[banda_num] / df[banda_den].replace(0, np.nan)
    return df

In [4]:
def codificar_fecha_ciclica(df, columna_fecha):
    df = df.copy()
    fecha = pd.to_datetime(df[columna_fecha])
    dia_anio = fecha.dt.dayofyear
    df["dia_anio_sin"] = np.sin(2 * np.pi * dia_anio / 365)
    df["dia_anio_cos"] = np.cos(2 * np.pi * dia_anio / 365)
    df["mes"] = fecha.dt.month
    return df

In [6]:
def promedio_vecindad_espacial(df, columna_valor, columna_lat, columna_lon,radio_grados=0.0005):
    from scipy.spatial import cKDTree

    coords = df[[columna_lat, columna_lon]].to_numpy()
    tree = cKDTree(coords)
    promedios = []
    for i, punto in enumerate(coords):
        vecinos = tree.query_ball_point(punto, r=radio_grados)
        promedios.append(df.iloc[vecinos][columna_valor].mean())

    df = df.copy()
    df[f"{columna_valor}_vecindad"] = promedios
    return df

In [7]:
def interaccion_ndvi_ndwi(df, col_ndvi="NDVI", col_ndwi="NDWI"):
    df = df.copy()
    df["NDVI_x_NDWI"] = df[col_ndvi] * df[col_ndwi]
    return df

In [8]:
def distancia_a_orilla(df, columna_lat, columna_lon, geom_orilla=None):
    raise NotImplementedError(
        "Pendiente: requiere la geometría de la orilla del lago (Ejercicio 1)."
    )